Plik generujący embeddingi MolFormer dla wybranych endpointów w celu wykorzystania ich do trenowania modeli.


In [8]:
!pip install numpy PyTDC transformers torch tqdm -q

In [1]:
import torch
from transformers import AutoModel, AutoTokenizer
from tdc.single_pred import ADME
import pandas as pd
import numpy as np
from tqdm import tqdm

#Ładowanie modelu MoLFormer
print("Ładowanie modelu MoLFormer...")
checkpoint = "ibm-research/MoLFormer-XL-both-10pct"

try:
    # Bardzo ważne: trust_remote_code=True pobiera skrypty modelujące z HF
    tokenizer = AutoTokenizer.from_pretrained(checkpoint, trust_remote_code=True)
    model = AutoModel.from_pretrained(checkpoint, trust_remote_code=True)

except Exception as e:
    print(f"Błąd: {e}")

def get_molformer_embedding(smiles_list, batch_size=32):
    all_embeddings = []

    for i in tqdm(range(0, len(smiles_list), batch_size)):
        batch_smiles = smiles_list[i:i+batch_size]

        # Tokenizacja
        inputs = tokenizer(batch_smiles, return_tensors="pt", padding=True, truncation=True).to(device) #zamienia smiles na wektory liczb (pt - python tensors)

        with torch.no_grad():#wyłączenie gradientów bo nie trenujemy modelu
            outputs = model(**inputs) #przepuszcza tokeny przez warstwy sieci MoLFormer

            # MoLFormer zwraca ukryte stany.
            # Najczęstszą metodą jest branie średniej z ostatniej warstwy (mean pooling) - zawsze da nam wektor o stałej długości
            embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            all_embeddings.append(embeddings)

    return np.vstack(all_embeddings)



Ładowanie modelu MoLFormer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_molformer_fast.py: 0.00B [00:00, ?B/s]

tokenization_molformer.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ibm-research/MoLFormer-XL-both-10pct:
- tokenization_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/ibm-research/MoLFormer-XL-both-10pct:
- tokenization_molformer_fast.py
- tokenization_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

configuration_molformer.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ibm-research/MoLFormer-XL-both-10pct:
- configuration_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_molformer.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ibm-research/MoLFormer-XL-both-10pct:
- modeling_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/187M [00:00<?, ?B/s]

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używam urządzenia: {device}")
model = model.to(device)
model.eval()

Używam urządzenia: cuda


MolformerModel(
  (embeddings): MolformerEmbeddings(
    (word_embeddings): Embedding(2362, 768, padding_idx=2)
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (encoder): MolformerEncoder(
    (layer): ModuleList(
      (0-11): 12 x MolformerLayer(
        (attention): MolformerAttention(
          (self): MolformerSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (rotary_embeddings): MolformerRotaryEmbedding()
            (feature_map): MolformerFeatureMap(
              (kernel): ReLU()
            )
          )
          (output): MolformerSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (in

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

# Definiowanie ścieżki do folderu
folder_path = './'

Edpoint 1:

In [5]:
# Ładowanie danych z TDC
data = ADME(name='Solubility_AqSolDB')
df = data.get_data()
# df zawiera kolumny 'Drug_ID', 'Drug' (SMILES), 'y' (rozpuszczalność)

Downloading...
100%|██████████| 853k/853k [00:00<00:00, 2.87MiB/s]
Loading...
Done!


In [6]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


100%|██████████| 312/312 [00:31<00:00,  9.86it/s]


In [7]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'Solubility_AqSolDB_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Wygenerowano embeddingi o rozmiarze: (9982, 768)
Plik został zapisany w: ./Solubility_AqSolDB_MoLFormer_embeddings.csv


Endpoint 2: Caco-2 (Wang)

In [ ]:
# Ładowanie danych z TDC
from tdc.single_pred import ADME
data = ADME(name='Caco2_Wang')
df = data.get_data()

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'Caco2_Wang_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 3: Lipophilicity (AstraZeneca)

In [ ]:
# Ładowanie danych z TDC
from tdc.single_pred import ADME
data = ADME(name='Lipophilicity_AstraZeneca')
df = data.get_data()

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'Lipophilicity_AstraZeneca_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 4: HIA (Hou)

In [ ]:
# Ładowanie danych z TDC
from tdc.single_pred import ADME
data = ADME(name='HIA_Hou')
df = data.get_data()

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'HIA_Hou_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 5: Half Life (Obach)

In [ ]:
# Ładowanie danych z TDC
from tdc.single_pred import ADME
data = ADME(name='Half_Life_Obach')
df = data.get_data()

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'Half_Life_Obach_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 6: Clearance Hepatocyte (AZ)

In [ ]:
# Ładowanie danych z TDC
from tdc.single_pred import ADME
data = ADME(name='Clearance_Hepatocyte_AZ')
df = data.get_data()

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'Clearance_Hepatocyte_AZ_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 7: CYP3A4 Inhibition (Veith)

In [ ]:
# Ładowanie danych z TDC
from tdc.single_pred import ADME
data = ADME(name='CYP3A4_Veith')
df = data.get_data()

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'CYP3A4_Veith_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 8: VDss (Lombardo)

In [ ]:
# Ładowanie danych z TDC
from tdc.single_pred import ADME
data = ADME(name='VDss_Lombardo')
df = data.get_data()

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'VDss_Lombardo_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 9: AMES Mutagenicity

In [ ]:
# Ładowanie danych z TDC
from tdc.single_pred import Tox
data = Tox(name='AMES')
df = data.get_data()

In [ ]:
#Generowanie
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)


In [ ]:
#zapis do pliku
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'AMES_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 10: hERG (Wang) - NEGATYWNY


In [ ]:
from tdc.single_pred import Tox

# Ładowanie danych z TDC
data = Tox(name='hERG')
df = data.get_data()

# Generowanie embeddingów
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)

# Konwersja embeddingów do DataFrame
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])

# Zapis do pliku
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'hERG_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endpoint 11: Pgp inhibition

In [ ]:
from tdc.single_pred import ADME

# Ładowanie danych z TDC
data = ADME(name='Pgp_Broccatelli')
df = data.get_data()

# Generowanie embeddingów
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)

# Konwersja embeddingów do DataFrame
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])

# Zapis do pliku
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'Pgp_Broccatelli_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

Endppoint 12: CYP2D6 Inhibition

In [ ]:
from tdc.single_pred import ADME

# Ładowanie danych z TDC
data = ADME(name='CYP2D6_Veith')
df = data.get_data()

# Generowanie embeddingów
smiles_list = df['Drug'].tolist()
embeddings = get_molformer_embedding(smiles_list)

# Konwersja embeddingów do DataFrame
df_embeddings = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings.shape[1])])

# Zapis do pliku
final_df = pd.concat([df[['Drug', 'Y']], df_embeddings], axis=1)

print(f"Wygenerowano embeddingi o rozmiarze: {embeddings.shape}")
file_path = os.path.join(folder_path, 'CYP2D6_Veith_MoLFormer_embeddings.csv')
final_df.to_csv(file_path, index=False)

print(f"Plik został zapisany w: {file_path}")

In [ ]:
# import pandas as pd
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import r2_score, mean_squared_error

# # 1. Wczytanie gotowego pliku
# folder_path = '/content/drive/MyDrive/MLDD - ADMET/STL_ML/embeddings'
# file_name = 'AqSolDB_MoLFormer_embeddings.csv'
# full_path = os.path.join(folder_path, file_name)

# print(f"Wczytywanie danych z pliku {file_name}...")
# data = pd.read_csv(full_path)

# # 2. Przygotowanie cech (X) i celu (y)
# # Wybieramy wszystkie kolumny, które zaczynają się od 'emb_'
# X = data.filter(like='emb_')

# # Szukamy kolumny celu (obsługujemy 'y' lub 'Y' w zależności od tego, jak zapisał się plik)
# target_col = 'Y' if 'Y' in data.columns else 'y'
# y = data[target_col]

# print(f"Wczytano {X.shape[0]} cząsteczek. Każda ma {X.shape[1]} cech (embeddingów).")

# # 3. Podział na zbiór treningowy i testowy (80/20)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # 4. Trening Random Forest
# print("Trenowanie modelu Random Forest...")
# # n_jobs=-1 wykorzystuje wszystkie rdzenie procesora w Colabie
# rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
# rf.fit(X_train, y_train)

# # 5. Ewaluacja
# preds = rf.predict(X_test)
# r2 = r2_score(y_test, preds)
# rmse = mean_squared_error(y_test, preds, squared=False)

# print("\n" + "="*30)
# print(f"WYNIKI DLA WCZYTANYCH EMBEDDINGÓW")
# print(f"R2 Score: {r2:.4f}")
# print(f"RMSE: {rmse:.4f}")
# print("="*30)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')